<a href="https://colab.research.google.com/github/princempatel2812-ship-it/Airline-Database/blob/main/Airline_database_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 18.5 MB/s eta 0:00:00


In [2]:
import sqlite3
import random
from faker import Faker

fake = Faker()

In [3]:
conn = sqlite3.connect("airline_database.db")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

In [5]:
cursor.executescript("""

CREATE TABLE IF NOT EXISTS Passengers (
    passenger_id INTEGER PRIMARY KEY,
    first_name TEXT,
    last_name TEXT,
    gender TEXT CHECK(gender IN ('Male','Female','Other')),
    age INTEGER CHECK(age BETWEEN 0 AND 100),
    nationality TEXT,
    frequent_flyer_status TEXT CHECK(frequent_flyer_status IN ('Bronze','Silver','Gold','Platinum')),
    registration_year INTEGER,
    passport_number TEXT
);

CREATE TABLE IF NOT EXISTS Aircraft (
    aircraft_id INTEGER PRIMARY KEY,
    model TEXT,
    capacity INTEGER,
    manufacturing_year INTEGER
);

CREATE TABLE IF NOT EXISTS Flights (
    flight_id INTEGER PRIMARY KEY,
    aircraft_id INTEGER,
    departure_city TEXT,
    arrival_city TEXT,
    flight_distance_km REAL,
    flight_status TEXT,
    departure_year INTEGER,
    FOREIGN KEY (aircraft_id) REFERENCES Aircraft(aircraft_id)
);

CREATE TABLE IF NOT EXISTS Bookings (
    booking_id INTEGER PRIMARY KEY,
    passenger_id INTEGER,
    booking_date TEXT,
    total_price REAL,
    booking_status TEXT,
    FOREIGN KEY (passenger_id) REFERENCES Passengers(passenger_id)
);

CREATE TABLE IF NOT EXISTS Tickets (
    booking_id INTEGER,
    flight_id INTEGER,
    seat_number TEXT,
    travel_class TEXT,
    baggage_weight REAL,
    ticket_price REAL,
    PRIMARY KEY (booking_id, flight_id),
    FOREIGN KEY (booking_id) REFERENCES Bookings(booking_id),
    FOREIGN KEY (flight_id) REFERENCES Flights(flight_id)
);

""")

In [7]:
for i in range(1200):
    passport = None if random.random() < 0.05 else fake.bothify(text="??######")
    if random.random() < 0.03:
        passport = "DUPLICATE123"

    cursor.execute("""
    INSERT INTO Passengers
    (first_name,last_name,gender,age,nationality,frequent_flyer_status,registration_year,passport_number)
    VALUES (?,?,?,?,?,?,?,?)
    """, (
        fake.first_name(),
        fake.last_name(),
        random.choice(['Male','Female','Other']),
        random.randint(18,80),
        fake.country(),
        random.choice(['Bronze','Silver','Gold','Platinum']),
        random.randint(2015,2025),
        passport
    ))

In [9]:
for i in range(50):
    cursor.execute("""
    INSERT INTO Aircraft (model,capacity,manufacturing_year)
    VALUES (?,?,?)
    """,(
        random.choice(['Boeing 737','Airbus A320','Boeing 787']),
        random.randint(150,350),
        random.randint(2000,2023)
    ))

In [10]:
for i in range(300):
    cursor.execute("""
    INSERT INTO Flights
    (aircraft_id,departure_city,arrival_city,flight_distance_km,flight_status,departure_year)
    VALUES (?,?,?,?,?,?)
    """,(
        random.randint(1,50),
        fake.city(),
        fake.city(),
        round(random.uniform(200,12000),2),
        random.choice(['Scheduled','Delayed','Cancelled','Completed']),
        random.randint(2018,2025)
    ))

In [11]:
for i in range(2000):
    cursor.execute("""
    INSERT INTO Bookings
    (passenger_id,booking_date,total_price,booking_status)
    VALUES (?,?,?,?)
    """,(
        random.randint(1,1200),
        fake.date_between(start_date='-3y', end_date='today'),
        round(random.uniform(50,2000),2),
        random.choice(['Confirmed','Cancelled','Pending'])
    ))

/tmp/ipython-input-344/2809696713.py:2: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute("""


In [12]:
for i in range(2000):
    cursor.execute("""
    INSERT OR IGNORE INTO Tickets
    (booking_id,flight_id,seat_number,travel_class,baggage_weight,ticket_price)
    VALUES (?,?,?,?,?,?)
    """,(
        random.randint(1,2000),
        random.randint(1,300),
        str(random.randint(1,30)) + random.choice(['A','B','C','D']),
        random.choice(['Economy','Business','First']),
        round(random.uniform(0,30),2),
        round(random.uniform(50,2000),2)
    ))

In [13]:
conn.commit()
conn.close()
print("Airline database created successfully")

Airline database created successfully
